# OTDR event detector — 4-class training

Классы: `bend`, `connector`, `break`, `background`.

- Реальные события берутся из ручных `.mask.json`.
- `background` — контролируемые отрицательные точки, удалённые от каждой ручной annotation.
- Split выполнен на уровне файлов, поэтому validation-трассы не видны модели в train.
- Главная практическая метрика: event-only macro F1 на `bend`/`connector`/`break`.
- Отдельно показывается macro F1 по всем 4 классам.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
EVENT_CLASSES = ['bend', 'connector', 'break']
ALL_CLASSES = EVENT_CLASSES + ['background']

FEATURE_COLS = [
    'm_norm', 'db_at_event', 'local_mean_db', 'local_std_db',
    'pre_mean_db', 'post_mean_db', 'loss_dB', 'peak_above_bg_dB',
    'pre_slope_dB_per_km', 'post_slope_dB_per_km', 'slope_change_dB_per_km',
    'derivative_at_event_dB_per_km', 'max_pre_derivative', 'min_post_derivative',
    'pre_std_db', 'post_std_db', 'post_to_pre_std_ratio',
    'peak_width_m', 'local_range_db'
]

DATA_DIR = Path.cwd() / 'gt_event_model_data_4class'
TRAIN_CSV = DATA_DIR / 'train_4class_events.csv'
VAL_CSV = DATA_DIR / 'val_4class_events.csv'
OUT_DIR = DATA_DIR / 'model_results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Data directory:', DATA_DIR)
print('Output directory:', OUT_DIR)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

missing = [c for c in FEATURE_COLS if c not in train_df.columns or c not in val_df.columns]
if missing:
    raise ValueError(f'Missing feature columns: {missing}')

train_df = train_df[train_df['label'].isin(ALL_CLASSES)].dropna(subset=FEATURE_COLS).copy()
val_df = val_df[val_df['label'].isin(ALL_CLASSES)].dropna(subset=FEATURE_COLS).copy()

X_train = train_df[FEATURE_COLS]
y_train = train_df['label']
X_val = val_df[FEATURE_COLS]
y_val = val_df['label']

print('TRAIN distribution:')
display(y_train.value_counts().reindex(ALL_CLASSES, fill_value=0).to_frame('count'))
print('VALIDATION distribution:')
display(y_val.value_counts().reindex(ALL_CLASSES, fill_value=0).to_frame('count'))

train_files = set(train_df['file'])
val_files = set(val_df['file'])
assert train_files.isdisjoint(val_files), 'Data leakage: a file exists in both train and val!'
print(f'No file leakage. Train files={len(train_files)}, validation files={len(val_files)}')

In [ ]:
models = {
    'DecisionTree': DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=4, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=500, max_depth=14, min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=500, max_depth=16, min_samples_leaf=2,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.04, max_depth=3, random_state=RANDOM_STATE
    ),
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=4000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'SVM_RBF': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced',
                    probability=True, random_state=RANDOM_STATE))
    ]),
}

min_class_n = int(y_train.value_counts().min())
n_splits = min(5, min_class_n)
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print(f'Cross-validation folds: {n_splits}; smallest class train count: {min_class_n}')

In [ ]:
results = []
fitted_models = {}
val_predictions = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    fitted = clone(model).fit(X_train, y_train)
    pred = fitted.predict(X_val)

    fitted_models[name] = fitted
    val_predictions[name] = pred

    p_all, r_all, f1_all, _ = precision_recall_fscore_support(
        y_val, pred, labels=ALL_CLASSES, average='macro', zero_division=0
    )
    p_event, r_event, f1_event, _ = precision_recall_fscore_support(
        y_val, pred, labels=EVENT_CLASSES, average='macro', zero_division=0
    )
    _, _, f1_weighted_all, _ = precision_recall_fscore_support(
        y_val, pred, labels=ALL_CLASSES, average='weighted', zero_division=0
    )

    row = {
        'model': name,
        'cv_f1_macro_all_mean': cv_scores.mean(),
        'cv_f1_macro_all_std': cv_scores.std(),
        'val_accuracy_all': accuracy_score(y_val, pred),
        'val_f1_macro_all': f1_all,
        'val_f1_weighted_all': f1_weighted_all,
        'val_precision_macro_events': p_event,
        'val_recall_macro_events': r_event,
        'val_f1_macro_events': f1_event,
    }

    per_p, per_r, per_f1, per_support = precision_recall_fscore_support(
        y_val, pred, labels=ALL_CLASSES, zero_division=0
    )
    for cls, precision, recall, f1, support in zip(ALL_CLASSES, per_p, per_r, per_f1, per_support):
        row[f'{cls}_precision'] = precision
        row[f'{cls}_recall'] = recall
        row[f'{cls}_f1'] = f1
        row[f'{cls}_support'] = support
    results.append(row)

comparison = pd.DataFrame(results).sort_values('val_f1_macro_events', ascending=False).reset_index(drop=True)
comparison.round(4)

In [ ]:
comparison.to_csv(OUT_DIR / 'model_comparison_4class.csv', index=False)

for name in comparison['model']:
    pred = val_predictions[name]
    print('=' * 90)
    print(name)
    print(classification_report(y_val, pred, labels=ALL_CLASSES, digits=3, zero_division=0))

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(confusion_matrix(y_val, pred, labels=ALL_CLASSES), display_labels=ALL_CLASSES).plot(
        ax=ax, cmap='Blues', values_format='d', colorbar=False
    )
    ax.set_title(f'{name} — validation confusion matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# Выбираем по event-only macro F1: background не должен искусственно сделать модель 'лучшей'.
best_name = comparison.iloc[0]['model']
best_model = fitted_models[best_name]

bundle = {
    'model': best_model,
    'model_name': best_name,
    'features': FEATURE_COLS,
    'classes': ALL_CLASSES,
    'event_classes': EVENT_CLASSES,
    'train_rows': len(train_df),
    'val_rows': len(val_df),
    'split': 'file-level; manual positives plus controlled background negatives',
    'selection_metric': 'validation macro F1 on bend/connector/break only',
    'best_val_f1_macro_events': float(comparison.iloc[0]['val_f1_macro_events']),
    'best_val_f1_macro_all': float(comparison.iloc[0]['val_f1_macro_all']),
}

model_path = OUT_DIR / 'best_4class_event_classifier.joblib'
joblib.dump(bundle, model_path)

print(f'Best model: {best_name}')
print(f'Validation event-only macro F1: {comparison.iloc[0]["val_f1_macro_events"]:.4f}')
print(f'Validation all-class macro F1: {comparison.iloc[0]["val_f1_macro_all"]:.4f}')
print(f'Saved: {model_path}')

In [ ]:
# Интерпретация tree-based winner: какие физические характеристики события наиболее полезны.
if best_name in {'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting'}:
    importance = pd.Series(best_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    display(importance.to_frame('importance'))
    fig, ax = plt.subplots(figsize=(9, 6))
    importance.sort_values().plot.barh(ax=ax, color='#1565C0')
    ax.set_title(f'{best_name}: feature importances')
    ax.set_xlabel('importance')
    plt.tight_layout()
    plt.show()
else:
    print(f'Feature importance is not directly available for {best_name}.')